In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Stage-I ybar inference (FULL-FOV tiled by default, exact inverse)
-----------------------------------------------------------------
✓ 默认全幅滑窗推理（无黑边、无中心小方块）
✓ 反归一化严格用仓库函数 normalize_image(..., invert=True)
✓ 支持可选直方图反转（--hist-eq）
✓ 保存时写入稳健 cal_min / cal_max（1%/99% 分位）=> 查看器不再“全黑”
✓ 可选 int16（--restored-int16），建议先用 float32 验证
✓ 唯一文件名（避免覆盖 recon）
"""

import os, sys, argparse, hashlib
from pathlib import Path, PureWindowsPath
import numpy as np, pandas as pd, nibabel as nb, torch
from tqdm import tqdm

# ---------- project import ----------
# 按你的环境：把工程根路径加入 import 路径
sys.path.append(r"/host/c/Users/ROG/Documents/Github")

from CTDenoising_Diffusion_N2N.Data_processing import (
    apply_transfer_to_img, cutoff_intensity, normalize_image, crop_or_pad
)

# ---------- Unet2D import ----------
def import_unet2d():
    try:
        from CTDenoising_Diffusion_N2N.noise2noise.model import Unet2D
        return Unet2D
    except Exception:
        import pkgutil, importlib, CTDenoising_Diffusion_N2N as _pkg
        for m in pkgutil.walk_packages(_pkg.__path__, _pkg.__name__ + "."):
            try:
                mod = importlib.import_module(m.name)
                if hasattr(mod, "Unet2D"):
                    print(f"[info] Fallback Unet2D from {m.name}")
                    return getattr(mod, "Unet2D")
            except Exception:
                pass
        raise ImportError("Cannot import Unet2D.")
Unet2D = import_unet2d()

# ---------- path mapping ----------
def map_win_to_posix(p: str, mount_root="/host"):
    """Map 'D:\\file\\...' -> '/host/d/file/...'; leave POSIX unchanged."""
    p = p.strip()
    if len(p) >= 2 and p[1] == ":":
        win = PureWindowsPath(p)
        drive = win.drive[:-1].lower()
        rest = str(win.relative_to(win.anchor)).replace("\\", "/")
        return f"{mount_root}/{drive}/{rest}"
    return p.replace("\\", "/")

# ---------- unique stem ----------
def make_stem(path: Path) -> str:
    """Compose unique output stem from parents to avoid overwriting 'recon'."""
    try:
        return "_".join([
            path.parent.parent.parent.name,   # patient
            path.parent.parent.name,          # sub
            path.parent.name,                 # random folder
            path.name.replace(".nii.gz","").replace(".nii",""),
        ]).replace(" ", "_")
    except Exception:
        h = hashlib.sha1(str(path).encode()).hexdigest()[:8]
        return path.stem + "_" + h

# ---------- LUT ----------
def load_luts(bins_path: Path, bins_mapped_path: Path, enable: bool):
    if not enable:
        return None, None
    if not bins_path.exists() or not bins_mapped_path.exists():
        raise FileNotFoundError(f"LUTs not found: {bins_path} or {bins_mapped_path}")
    return (np.load(str(bins_path), allow_pickle=True),
            np.load(str(bins_mapped_path), allow_pickle=True))

# ---------- preprocessing ----------
def preprocess_volume(vol, H, W, hist_eq, bins, bins_mapped, bg_cut, max_cut, norm, *, resize_xy: bool):
    """
    训练一致的预处理：
      1) 可选直方图映射
      2) 强制剪裁到 [bg_cut, max_cut]
      3) 归一化（仓库函数；训练用零中心）
      4) 可选 XY 裁剪/填充到 [H,W]（Z 不动）
    """
    if hist_eq:
        vol = apply_transfer_to_img(vol, bins, bins_mapped)
    vol = cutoff_intensity(vol, cutoff_low=bg_cut, cutoff_high=max_cut)
    vol = normalize_image(vol, normalize_factor=norm, image_max=max_cut, image_min=bg_cut, invert=False)
    if resize_xy:
        vol = crop_or_pad(vol, [H, W, vol.shape[2]], value=float(vol.min()))
    return vol

# ---------- inference (single window) ----------
@torch.no_grad()
def run_n2n_on_window(model, win_norm, batch_z=32):
    """
    在一个 XY window 上做推理（使用两侧切片做2通道输入）
    win_norm: [H,W,Z] -> out: [H,W,Z]
    """
    device = next(model.parameters()).device
    H0, W0, Z = map(int, win_norm.shape)
    z = np.arange(Z)
    z0, z2 = np.maximum(z-1,0), np.minimum(z+1,Z-1)

    a = np.ascontiguousarray(win_norm[:, :, z0])
    b = np.ascontiguousarray(win_norm[:, :, z2])
    pairs = np.transpose(np.stack([a, b], axis=0), (3, 0, 1, 2))  # [Z,2,H,W]

    t = torch.from_numpy(pairs).to(device)
    # 让输入 dtype 跟模型参数匹配
    if next(model.parameters()).dtype == torch.float16:
        t = t.half()
    else:
        t = t.float()

    y = np.zeros((H0, W0, Z), np.float32)
    for i in range(0, Z, batch_z):
        j = min(i + batch_z, Z)
        out = model(t[i:j])  # [Bz,1,H,W]
        out_np = out.squeeze(1).detach().cpu().float().numpy()  # [Bz,H,W]
        y[:, :, i:j] = np.transpose(out_np, (1, 2, 0))
    return np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)

# ---------- tiled full-FOV inference ----------
def _make_starts(L, tile, stride):
    """Start indices covering [0, L) with last window anchored to end."""
    if L <= tile:
        return [0]
    starts = list(range(0, max(1, L - tile + 1), max(1, stride)))
    if starts[-1] != L - tile:
        starts.append(L - tile)
    return starts

@torch.no_grad()
def run_n2n_tiled(model, vol_norm_full, tile_h, tile_w, stride, batch_z=32):
    """
    整幅图 XY 滑窗；重叠区做平均
    vol_norm_full: [H0,W0,Z] -> returns: [H0,W0,Z]
    """
    H0, W0, Z = map(int, vol_norm_full.shape)
    y_full = np.zeros((H0, W0, Z), np.float32)
    w_full = np.zeros((H0, W0, Z), np.float32)

    hs = _make_starts(H0, tile_h, stride)
    ws = _make_starts(W0, tile_w, stride)

    pad_val = float(vol_norm_full.min())  # 边界 pad 用当前窗的背景最小值

    for y0 in hs:
        for x0 in ws:
            y1, x1 = y0 + tile_h, x0 + tile_w
            patch = vol_norm_full[y0:y1, x0:x1, :]
            ph, pw = patch.shape[0], patch.shape[1]
            if ph != tile_h or pw != tile_w:
                tmp = np.full((tile_h, tile_w, Z), pad_val, dtype=patch.dtype)
                tmp[:ph, :pw, :] = patch
                patch = tmp

            y_win = run_n2n_on_window(model, patch, batch_z=batch_z)  # [tile_h,tile_w,Z]

            vh, vw = min(tile_h, H0 - y0), min(tile_w, W0 - x0)
            y_full[y0:y0+vh, x0:x0+vw, :] += y_win[:vh, :vw, :]
            w_full[y0:y0+vh, x0:x0+vw, :] += 1.0

    w_full[w_full == 0] = 1.0
    return (y_full / w_full).astype(np.float32)

# ---------- load model ----------
def load_model(ckpt: Path, use_half: bool, try_compile=False):
    """
    IMPORTANT: channels=2 因为我们喂的是 [前一/后一] 邻居两通道
    """
    model = Unet2D(channels=2, out_dim=1, init_dim=16, dim_mults=(2, 4, 8, 16))
    state = torch.load(str(ckpt), map_location="cpu")

    if isinstance(state, dict) and "model" in state:
        model.load_state_dict(state["model"], strict=False)
        if "ema" in state:
            try:
                from ema_pytorch import EMA
                ema = EMA(model); ema.load_state_dict(state["ema"])
                model = ema.ema_model
                print("[info] Loaded EMA weights (EMA model).")
            except Exception:
                pass
    else:
        model.load_state_dict(state, strict=False)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device).eval()
    for p in model.parameters():
        p.requires_grad = False

    if use_half and device == "cuda":
        model = model.half()
        torch.backends.cudnn.benchmark = True
        try:
            torch.set_float32_matmul_precision("high")
        except Exception:
            pass

    if try_compile:
        try:
            model = torch.compile(model)
            print("[info] torch.compile enabled.")
        except Exception as e:
            print(f"[warn] compile failed: {e}")
    return model

# ---------- subset select ----------
def select_subset(df, fold, split, use_randoms=(0, 1)):
    if "batch" not in df.columns:
        raise RuntimeError("Excel must have 'batch'.")
    batches = sorted(df["batch"].dropna().astype(int).unique().tolist())
    K = len(batches)
    if fold < 0 or fold >= K:
        raise ValueError(f"fold out of range: {batches}")
    test_b = batches[fold]
    val_b  = batches[(fold + 1) % K]
    if   split == "train":    df_use = df[~df["batch"].astype(int).isin([test_b, val_b])]
    elif split == "val":      df_use = df[df["batch"].astype(int) == val_b]
    elif split == "test":     df_use = df[df["batch"].astype(int) == test_b]
    elif split == "trainval": df_use = df[~df["batch"].astype(int).isin([test_b])]
    else:
        raise ValueError("split must be train/val/test/trainval")
    if "random_num" in df_use.columns and use_randoms is not None:
        df_use = df_use[df_use["random_num"].isin(list(use_randoms))]
    return df_use

# ---------- exact inverse to display space ----------
def inverse_to_display_space(ybar, vol_orig, hist_eq, bins, bins_mapped, bg, mx, norm):
    """
    反归一化严格使用仓库的 normalize_image(..., invert=True)；
    再可选反直方图；最后把 XY 对齐回原图。
    """
    # 1) exact inverse
    y = normalize_image(
        ybar.astype(np.float32),
        normalize_factor=norm,
        image_max=mx,
        image_min=bg,
        invert=True
    ).astype(np.float32)

    # 2) reverse histogram (if enabled)
    if hist_eq and (bins is not None) and (bins_mapped is not None):
        try:
            y = apply_transfer_to_img(y, bins, bins_mapped, reverse=True)
        except TypeError:
            pass

    # 3) XY match to original
    H0, W0, _ = vol_orig.shape
    if y.shape[0] != H0 or y.shape[1] != W0:
        y = crop_or_pad(y, [H0, W0, y.shape[2]], value=float(np.nanmin(vol_orig)))

    return y

# ---------- main ----------
def main(argv=None):
    parser = argparse.ArgumentParser(allow_abbrev=False)
    # paths
    parser.add_argument("--excel",   type=str, default=r"/host/d/file/fixedCT_static_simulation_train_test_gaussian_local.xlsx")
    parser.add_argument("--ckpt",    type=str, default=r"/host/d/file/noise2noise/model-78.pt")
    parser.add_argument("--outdir",  type=str, default=r"/host/d/file/outputs_ybar")
    # split
    parser.add_argument("--fold",    type=int, default=0)
    parser.add_argument("--split",   type=str, default="train", choices=["train", "val", "test", "trainval"])
    parser.add_argument("--randoms", type=int, nargs="*", default=[0, 1])
    # preprocess
    parser.add_argument("--H", type=int, default=256)
    parser.add_argument("--W", type=int, default=256)
    parser.add_argument("--hist-eq", action="store_true")
    parser.add_argument("--bins",        type=str, default=r"/host/d/file/histogram_equalization/bins.npy")
    parser.add_argument("--bins-mapped", type=str, default=r"/host/d/file/histogram_equalization/bins_mapped.npy")
    parser.add_argument("--bg-cut", type=float, default=0.0)
    parser.add_argument("--max-cut", type=float, default=2000.0)
    parser.add_argument("--norm",   type=float, default=1000.0)
    # speed / precision
    parser.add_argument("--batch-z", type=int, default=32)
    parser.add_argument("--no-half", action="store_true")
    parser.add_argument("--compile", action="store_true")
    # overwrite & saved types
    ow = parser.add_mutually_exclusive_group()
    ow.add_argument("--overwrite",    dest="overwrite", action="store_true")
    ow.add_argument("--no-overwrite", dest="overwrite", action="store_false")
    parser.set_defaults(overwrite=True)
    parser.add_argument("--no-save-restored", dest="save_restored", action="store_false")
    parser.set_defaults(save_restored=True)
    parser.add_argument("--restored-int16", action="store_true", help="store restored as int16 on disk")
    parser.add_argument("--debug-first-k", type=int, default=1)
    # tiled inference options (DEFAULT = tiled full-FOV)
    tile_group = parser.add_mutually_exclusive_group()
    tile_group.add_argument("--tile",    dest="tile", action="store_true",
                            help="(default) enable sliding-window full-FOV inference")
    tile_group.add_argument("--no-tile", dest="tile", action="store_false",
                            help="disable tiled inference; use center HxW window")
    parser.set_defaults(tile=True)
    parser.add_argument("--tile-size",   type=int, default=256, help="model window size (H=W)")
    parser.add_argument("--tile-stride", type=int, default=192, help="stride；overlap = size - stride")

    # jupyter 友好：去掉 notebook 注入的 -f 参数
    if argv is None and 'ipykernel' in sys.modules:
        argv = []
    args = parser.parse_args(argv)

    excel  = Path(args.excel)
    outdir = Path(args.outdir); outdir.mkdir(parents=True, exist_ok=True)

    cuda = torch.cuda.is_available()
    dev  = torch.cuda.get_device_name(0) if cuda else "CPU"
    print(f"[env] CUDA={cuda} | Device={dev}")

    df = pd.read_excel(str(excel))
    if "noise_file" not in df.columns:
        raise RuntimeError("Excel must contain 'noise_file'.")
    df["noise_file_mapped"] = df["noise_file"].astype(str).apply(map_win_to_posix)
    df_use = select_subset(df, args.fold, args.split, args.randoms)

    all_paths = df_use["noise_file_mapped"].dropna().astype(str).unique().tolist()
    paths = [p for p in all_paths if os.path.exists(p)]
    miss  = sorted(set(all_paths) - set(paths))
    print(f"[split] fold={args.fold}, split={args.split}, total={len(all_paths)}, accessible={len(paths)}, missing={len(miss)}")
    if miss:
        print("[warn] missing example:", miss[0])

    bins, bins_mapped = load_luts(Path(args.bins), Path(args.bins_mapped), args.hist_eq)
    use_half = (not args.no_half)
    model = load_model(Path(args.ckpt), use_half=use_half, try_compile=args.compile)

    saved=failed=skipped=0
    dbg_left = args.debug_first_k

    for p in tqdm(paths, desc="ybar inference"):
        try:
            vol_path = Path(p)
            stem = make_stem(vol_path)
            out_model = outdir / f"{stem}_ybar.nii.gz"
            out_disp  = outdir / f"{stem}_ybar_restored.nii.gz"

            if out_model.exists() and out_disp.exists() and not args.overwrite:
                skipped += 1
                continue

            nii = nb.load(str(vol_path))
            vol = nii.get_fdata().astype(np.float32)  # [H0,W0,Z]

            if args.tile:
                # 全幅：预处理不改变 XY
                vol_n = preprocess_volume(
                    vol, args.H, args.W,
                    args.hist_eq, bins, bins_mapped,
                    args.bg_cut, args.max_cut, args.norm,
                    resize_xy=False
                )
                ybar = run_n2n_tiled(
                    model, vol_n,
                    tile_h=args.tile_size, tile_w=args.tile_size,
                    stride=args.tile_stride,
                    batch_z=args.batch_z
                )  # [H0,W0,Z]
            else:
                # 单窗：中心裁剪/填充到 HxW
                vol_n = preprocess_volume(
                    vol, args.H, args.W,
                    args.hist_eq, bins, bins_mapped,
                    args_bg_cut, args.max_cut, args.norm,
                    resize_xy=True
                )
                ybar = run_n2n_on_window(model, vol_n, batch_z=args.batch_z)  # [H,W,Z]

            # 保存 ybar（model space）
            img_y = nb.Nifti1Image(ybar, nii.affine)
            hdr_y = img_y.header
            hdr_y["scl_slope"] = 1.0; hdr_y["scl_inter"] = 0.0
            hdr_y["cal_min"]   = float(np.nanmin(ybar))
            hdr_y["cal_max"]   = float(np.nanmax(ybar))
            nb.save(img_y, str(out_model))

            # 反归一化 + 反直方图 + XY 对齐（精确反向）
            y_full = inverse_to_display_space(
                ybar, vol,
                args.hist_eq, bins, bins_mapped,
                args.bg_cut, args.max_cut, args.norm
            )

            if args.save_restored:
                if args.restored_int16:
                    # 建议：先 float32 验证；若用 int16，可在实际项目里写 slope/intercept 做物理单位映射
                    y16 = np.clip(y_full, args.bg_cut, args.max_cut)
                    y16 = np.round(y16).astype(np.int16)
                    img_r = nb.Nifti1Image(y16, nii.affine)
                else:
                    img_r = nb.Nifti1Image(y_full.astype(np.float32), nii.affine)

                hdr_r = img_r.header
                hdr_r["scl_slope"] = 1.0; hdr_r["scl_inter"] = 0.0
                # 用 1% / 99% 分位，查看器窗口更稳，不会“一片黑”
                p1, p99 = np.percentile(y_full, [1, 99])
                hdr_r["cal_min"] = float(p1)
                hdr_r["cal_max"] = float(p99)
                nb.save(img_r, str(out_disp))

            if dbg_left > 0:
                vcut = np.clip(vol, args.bg_cut, args.max_cut)
                p1, p50, p99 = [float(x) for x in np.percentile(y_full, [1,50,99])]
                print(f"[dbg] {stem}")
                print("  orig after cut  :", float(vcut.min()), float(vcut.max()))
                print("  ybar (model)    :", float(ybar.min()), float(ybar.max()))
                print("  restored p1/50/99:", p1, p50, p99)
                dbg_left -= 1

            saved += 1

        except Exception as e:
            print(f"[skip] {p} failed: {e}")
            failed += 1

    print(f"✅ Done. saved={saved}, failed={failed}, skipped={skipped} -> {outdir}")

# 兼容你在 Notebook 里像模块那样调用：ybar.main([...])
if __name__ == "__main__":
    main()






[env] CUDA=True | Device=NVIDIA GeForce RTX 4080 Laptop GPU
[split] fold=0, split=train, total=132, accessible=132, missing=0
in out is :  [(16, 32), (32, 64), (64, 128), (128, 256)]


ybar inference:   1%|          | 1/132 [00:17<37:23, 17.12s/it]

[dbg] 00214783_0000455264_gaussian_random_0_recon
  orig after cut  : 0.0 1575.95068359375
  mode            : tiled
  ybar (model)    : -1.052734375 0.56494140625


ybar inference:  39%|███▉      | 52/132 [15:15<23:29, 17.61s/it]


KeyboardInterrupt: 